In [3]:
from WindPy import w
import numpy as np
import pandas as pd
import datetime
import os
import json
from tqdm import tqdm

w.start()
w.isconnected()

2025-05-03 20:44:17.195 Python[59852:2841590] WARNING: Secure coding is not enabled for restorable state! Enable secure coding by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState: and returning YES.


25.1.5.60030
2025-05-03 20:44:23.585 [Sys] INFO  Using UTF8 Encodinglogger id=0 key=Main name=MainLog path=/Users/Bill/.Wind/WFT/log/TBAPI2log/ level=2 display=1 
Wind.Cosmos.Base V1.8 compiled time is Dec 18 2024, BuildType:Release, CPUArch:X64, GCC Version:Apple LLVM 13.0.0 (clang-1300.0.29.30)
Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2021 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [4]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)

In [5]:
df = pd.read_csv("A股指数.csv")
a_share_dict = dict(zip(df["Ticker"], df["Name"]))

df = pd.read_csv("海外指数.csv")
other_market_dict = dict(zip(df["Ticker"], df["Name"]))

print(len(a_share_dict), len(other_market_dict))

333 57


In [6]:
len(INDEX_START)

189

# Load Local Database

In [7]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

186

In [8]:
print(INDEX_DATA['000300.SH'].index[-1])

2025-04-28


# Update Existing Tickers

In [10]:
today = "2025-04-28"

In [9]:
for ticker in tqdm(INDEX_DATA):
    data = INDEX_DATA[ticker]
    start = (data.index[-1] + pd.Timedelta(days = 1)).strftime("%Y-%m-%d")
    
    new_data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]
    
    if new_data.columns[0] == 'OUTMESSAGE':
        print(ticker)
        continue
    
    INDEX_DATA[ticker] = pd.concat([data, new_data])
    INDEX_DATA[ticker].dropna(inplace = True)

100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [00:50<00:00,  1.38it/s]


In [6]:
INDEX_DATA['000300.SH'].tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-04-22,12.2931,3.5380,3783.9522
2025-04-23,12.2687,3.5279,3786.8824
2025-04-24,12.2906,3.5176,3784.3574
2025-04-25,12.2609,3.5798,3786.9937
2025-04-28,12.3204,3.5752,3781.6188


# Download New Tickers

In [11]:
for ticker, start in tqdm(INDEX_START.items()):
    if ticker in INDEX_DATA:
        continue
        
    assert ticker not in INDEX_DATA
    
    data = w.wsd(ticker, 'pe_ttm,dividendyield2,close', start, today, '', usedf = True)[1]
    
    junk = data['PE_TTM']

    if data.isna().sum().sum() != 0:
        print(ticker)
    
    data.dropna(inplace = True)
    
    INDEX_DATA[ticker] = data.copy(deep = True)

 99%|███████████████████████████████████████▌| 187/189 [00:00<00:00, 270.33it/s]

SP6CSSUP.SPI


100%|█████████████████████████████████████████| 189/189 [00:02<00:00, 84.13it/s]


In [13]:
for ticker, data in INDEX_DATA.items():
    path = os.path.join("Data", f"{ticker}.csv")
    data.to_csv(path, index = True, encoding = "utf-8")

In [12]:
len(INDEX_DATA)

131

In [12]:
temp = INDEX_DATA['SP6CSSUP.SPI']

In [13]:
temp.head()

,PE_TTM,DIVIDENDYIELD2,CLOSE


In [14]:
INDEX_START['SP6CSSUP.SPI']

'2014-01-01'